# Exercise 2: 计算图实现用于 TinyML 唤醒词检测

## MAIE 5532: 机器学习系统 - 第 2 周

### 学习目标：
• 理解反向模式 AD 中的计算图构建
• 实现前向传播，并有策略地保存中间值
• 实现完整反向传播以计算梯度
• 应用嵌入式系统中的内存管理技巧
• 将理论与实际的唤醒词检测系统联系起来

### 🎯 什么是计算图？

计算图是反向模式自动微分的基础数据结构。把它想象成一个“记住每一步运算”的配方：

- **节点**：表示数值（输入、参数、中间结果、输出）
- **边**：表示将值变换为新值的运算
- **前向传播**：按照配方进行计算，并保存中间“原料”
- **反向传播**：反向执行配方，找出每个“原料”对最终结果的影响

### 为什么这对唤醒词检测很重要？
每当智能音箱识别“Hey Siri”或“OK Google”时，它都在使用由计算图训练出来的神经网络。通过这些计算图计算得到的梯度，使系统能够从海量语音样本中学习。

### 现实世界中的影响：
- **隐私**：端侧学习意味着你的语音不会离开设备
- **个性化**：模型会适应你的特定语音模式
- **效率**：为微控制器部署而优化
- **联邦学习**：在不共享数据的前提下参与全球模型改进

In [1]:
# 计算图实现所需的基础导入
import math
import numpy as np
from IPython.core.display import HTML

# 用于干净输出显示的工具函数
def show(title, *pairs):
    """
    以结构化方式展示结果的辅助函数
    这让输出更易读且更专业
    """
    print(title)
    for k, v in pairs:
        print(f"  {k}: {v}")

print("🚀 练习 2：用于唤醒词检测的计算图")
print("=" * 60)
print()
print("✅ 导入成功！")
print()
print("📝 代码说明：")
print("  • numpy：用于高效矩阵运算")
print("  • math：基础数学函数")
print("  • show()：用于干净结果展示")
print()
print("🧠 计算图概念：")
print("  计算图会跟踪神经网络中的每一步运算")
print("  就像一份详细的食谱，记住每一道烹饪步骤")
print("  前向传播：照着食谱做，反向传播：反向推导每一步原因")

🚀 练习 2：用于唤醒词检测的计算图

✅ 导入成功！

📝 代码说明：
  • numpy：用于高效矩阵运算
  • math：基础数学函数
  • show()：用于干净结果展示

🧠 计算图概念：
  计算图会跟踪神经网络中的每一步运算
  就像一份详细的食谱，记住每一道烹饪步骤
  前向传播：照着食谱做，反向传播：反向推导每一步原因


## 🏗️ 构建计算图基础设施

### 节点设计理念

我们的 `ComputationNode` 类是图的基本构件。每个节点都需要保存：

1. **值**：前向传播中实际计算出的数值
2. **梯度**：在反向传播中该节点相对于最终输出的变化程度
3. **依赖关系**：该节点依赖哪些父节点（图结构）
4. **反向函数**：用于计算该运算对应梯度的函数

### 内存管理策略

在嵌入式系统中，我们必须非常注意内存使用：
- **预分配**：在编译期就知道内存需求
- **有策略的存储**：只保留反向传播绝对需要的内容
- **内存池**：尽可能复用内存位置
- **混合精度**：尽量使用 16 位，在必要时使用 32 位

### 反向模式的魔法

与前向模式不同（它一次只计算一个输入的导数），反向模式：
- ✅ 一次反向传播即可计算所有参数梯度
- ✅ 对神经网络非常高效（参数很多，但只有一个损失值）
- ⚠️ 需要保存中间值，因此内存成本更高
- 🎯 非常适合训练场景，对仅推理的嵌入式系统更具挑战

In [2]:
class ComputationNode:
    """
    计算图中的一个节点，用于反向模式自动微分
    
    它就像一个“智能容器”，保存：
    - 前向传播计算得到的数值
    - 反向传播中计算出的梯度
    - 对父节点的引用（依赖关系）
    - 计算梯度的函数（反向函数）
    
    可以把它想成一条配方步骤，既记得结果，
    也记得如何反向还原以计算梯度。
    """
    
    def __init__(self, value, name="", requires_grad=True):
        """
        创建一个计算节点
        
        Args:
            value: 数值（标量、向量或矩阵）
            name: 人类可读的标识符，用于调试
            requires_grad: 是否需要梯度计算
        """
        self.value = np.array(value, dtype=np.float32)
        self.gradient = np.zeros_like(self.value)
        self.name = name
        self.requires_grad = requires_grad
        self.backward_fn = None
        self.inputs = []
        print(f"📦 创建节点 '{self.name}'：")
        print(f"    形状: {self.value.shape}")
        print(f"    值: {self.value}")
        print(f"    是否需要梯度: {requires_grad}")
    
    def __repr__(self):
        return f"Node('{self.name}', shape={self.value.shape}, grad_norm={np.linalg.norm(self.gradient):.6f})"

print("✅ ComputationNode 类已实现！")
print()
print("🔍 我们刚刚构建了什么：")
print("  • 一个用于保存数值和梯度的‘智能容器’")
print("  • 使用 numpy 数组进行内存高效存储")
print("  • 使用输入引用追踪图结构")
print("  • 支持名称和表示输出的调试功能")
print()
print("💡 关键洞察：")
print("  每个节点都像一条食谱步骤，既记得结果")
print("  也记得如何反向计算梯度以保证梯度流动")

✅ ComputationNode 类已实现！

🔍 我们刚刚构建了什么：
  • 一个用于保存数值和梯度的‘智能容器’
  • 使用 numpy 数组进行内存高效存储
  • 使用输入引用追踪图结构
  • 支持名称和表示输出的调试功能

💡 关键洞察：
  每个节点都像一条食谱步骤，既记得结果
  也记得如何反向计算梯度以保证梯度流动


## 🧮 实现神经网络运算

### 线性层：神经网络中的主力

线性（全连接）层是基础模块：**output = input @ weight.T + bias**

**前向传播数学：**
- 矩阵乘法：利用学习到的权重组合输入特征
- 偏置相加：移动决策边界
- 结果：生成下一层可使用的特征表示

**反向传播数学（链式法则）：**
- ∂Loss/∂input = ∂Loss/∂output @ weight
- ∂Loss/∂weight = ∂Loss/∂output.T @ input
- ∂Loss/∂bias = sum(∂Loss/∂output)

### 为什么采用这种实现策略？
1. **模块化**：每个运算都是独立封装
2. **可组合性**：多个运算可以串联起来
3. **自动梯度**：反向函数自动处理链式法则
4. **内存效率**：只保存反向传播所需的信息

In [3]:
def linear_layer(input_node, weight_node, bias_node, name="linear"):
    """
    线性（全连接）层：output = input @ weight.T + bias
    
    这是神经网络中最基础的构建模块。
    我们同时进行前向计算并设置反向传播函数。
    """
    print(f"\n🔄 计算 {name} 层")
    print(f"    输入形状: {input_node.value.shape}")
    print(f"    权重形状: {weight_node.value.shape}")
    print(f"    偏置形状: {bias_node.value.shape}")
    matmul_result = input_node.value @ weight_node.value.T
    linear_output = matmul_result + bias_node.value
    output_node = ComputationNode(linear_output, name=f"{name}_output")
    output_node.inputs = [input_node, weight_node, bias_node]

    def backward():
        print(f"\n⬅️ {name} 的反向传播")
        if input_node.requires_grad:
            input_grad = output_node.gradient @ weight_node.value
            input_node.gradient += input_grad
        if weight_node.requires_grad:
            grad_reshaped = output_node.gradient.reshape(-1, 1) if output_node.gradient.ndim == 1 else output_node.gradient
            input_reshaped = input_node.value.reshape(1, -1) if input_node.value.ndim == 1 else input_node.value
            weight_grad = grad_reshaped @ input_reshaped
            weight_node.gradient += weight_grad
        if bias_node.requires_grad:
            bias_grad = output_node.gradient.copy()
            bias_node.gradient += bias_grad
    output_node.backward_fn = backward
    return output_node

print("✅ 线性层实现完成！")

✅ 线性层实现完成！


## 🎛️ 激活函数：非线性与梯度流动

### Tanh 激活：平滑非线性

**数学性质：**
- **范围**：(-1, 1) - 保持数值有界
- **导数**：1 - tanh²(x) - 可直接由前向值计算
- **零中心化**：有助于深层网络中梯度流动

### ReLU：深度学习革命

**数学性质：**
- **函数**：max(0, x)
- **导数**：若 x > 0，则为 1，否则为 0
- **计算效率**：非常高，只需比较大小

### Softmax：概率分布

**数学性质：**
- **函数**：exp(xi) / sum(exp(x))
- **输出**：有效的概率分布（总和为 1）
- **导数**：与交叉熵损失结合时非常优雅

In [4]:
def tanh_activation(input_node, name="tanh"):
    tanh_output = np.tanh(input_node.value)
    output_node = ComputationNode(tanh_output, name=f"{name}_output")
    output_node.inputs = [input_node]
    def backward():
        if input_node.requires_grad:
            tanh_derivative = 1 - output_node.value ** 2
            input_grad = output_node.gradient * tanh_derivative
            input_node.gradient += input_grad
    output_node.backward_fn = backward
    return output_node

def softmax_activation(input_node, name="softmax"):
    max_val = np.max(input_node.value)
    stable_input = input_node.value - max_val
    exp_values = np.exp(stable_input)
    softmax_output = exp_values / np.sum(exp_values)
    output_node = ComputationNode(softmax_output, name=f"{name}_output")
    output_node.inputs = [input_node]
    def backward():
        if input_node.requires_grad:
            s = output_node.value.reshape(-1, 1)
            jacobian = np.diagflat(s) - np.dot(s, s.T)
            input_grad = jacobian @ output_node.gradient.reshape(-1, 1)
            input_node.gradient += input_grad.flatten()
    output_node.backward_fn = backward
    return output_node

print("✅ 激活函数已实现！")

✅ 激活函数已实现！


## 🎯 损失函数：用于分类的交叉熵

### 为什么使用交叉熵损失？

交叉熵损失是分类任务的黄金标准，因为：

1. **概率解释**：衡量预测概率与真实分布之间的距离
2. **梯度性质**：当预测错误时会提供强梯度
3. **数学优雅**：与 softmax 激活天然配合
4. **凸性**：对线性模型具有良好的优化性质

### 数学上的美妙之处

**前向**：Loss = -∑(target_i × log(prediction_i))
**反向**（与 softmax 联合）：∂Loss/∂logits = predictions - targets

这也是为什么 softmax + cross-entropy 在深度学习中如此普遍。

In [5]:
def cross_entropy_loss(predictions_node, target_node, name="cross_entropy"):
    epsilon = 1e-15
    safe_predictions = np.clip(predictions_node.value, epsilon, 1 - epsilon)
    log_predictions = np.log(safe_predictions)
    loss_terms = target_node.value * log_predictions
    loss_value = -np.sum(loss_terms)
    loss_node = ComputationNode(loss_value, name=f"{name}_output")
    loss_node.inputs = [predictions_node, target_node]
    def backward():
        if predictions_node.requires_grad:
            pred_grad = -target_node.value / safe_predictions
            predictions_node.gradient += pred_grad
    loss_node.backward_fn = backward
    return loss_node

print("✅ 交叉熵损失已实现！")

✅ 交叉熵损失已实现！


## 🎵 唤醒词检测网络实现

### 网络架构：4 → 3 → 2

我们的唤醒词检测网络采用了精心设计的架构：

**输入层（4 个特征）**：
- 频谱特征（主导频率、谱质心）
- 能量特征（RMS 能量、过零率）
- 时间特征（时长、停顿检测）
- 振幅特征（峰值幅度、动态范围）

**隐藏层（3 个神经元）**：
- 学习复杂特征组合
- 使用 tanh 激活，梯度更平滑
- 容量足够处理简单唤醒词模式

**输出层（2 个类别）**：
- 类别 0：检测到唤醒词
- 类别 1：未检测到唤醒词（背景/静音）
- 使用 softmax 激活生成概率分布

In [6]:
# 唤醒词检测网络设置
print("🎵 唤醒词检测网络")
print("=" * 50)
print()
audio_features = [0.2, -0.1, 0.5, 0.3]
target = [1, 0]

input_node = ComputationNode(audio_features, name="audio_input", requires_grad=False)
target_node = ComputationNode(target, name="target", requires_grad=False)

np.random.seed(42)
W1 = np.random.randn(3, 4).astype(np.float32) * 0.1
b1 = np.zeros(3, dtype=np.float32)
W2 = np.random.randn(2, 3).astype(np.float32) * 0.1
b2 = np.zeros(2, dtype=np.float32)

W1_node = ComputationNode(W1, name="W1", requires_grad=True)
b1_node = ComputationNode(b1, name="b1", requires_grad=True)
W2_node = ComputationNode(W2, name="W2", requires_grad=True)
b2_node = ComputationNode(b2, name="b2", requires_grad=True)

z1_node = linear_layer(input_node, W1_node, b1_node, name="layer1_linear")
a1_node = tanh_activation(z1_node, name="layer1_activation")
z2_node = linear_layer(a1_node, W2_node, b2_node, name="layer2_linear")
probs_node = softmax_activation(z2_node, name="layer2_softmax")
loss_node = cross_entropy_loss(probs_node, target_node, name="training_loss")

print("✅ 网络前向传播已构建完成")

🎵 唤醒词检测网络

📦 创建节点 'audio_input'：
    形状: (4,)
    值: [ 0.2 -0.1  0.5  0.3]
    是否需要梯度: False
📦 创建节点 'target'：
    形状: (2,)
    值: [1. 0.]
    是否需要梯度: False
📦 创建节点 'W1'：
    形状: (3, 4)
    值: [[ 0.04967142 -0.01382643  0.06476886  0.15230298]
 [-0.02341534 -0.0234137   0.15792128  0.07674348]
 [-0.04694744  0.054256   -0.04634177 -0.04657298]]
    是否需要梯度: True
📦 创建节点 'b1'：
    形状: (3,)
    值: [0. 0. 0.]
    是否需要梯度: True
📦 创建节点 'W2'：
    形状: (2, 3)
    值: [[ 0.02419623 -0.19132803 -0.17249179]
 [-0.05622875 -0.10128311  0.03142473]]
    是否需要梯度: True
📦 创建节点 'b2'：
    形状: (2,)
    值: [0. 0.]
    是否需要梯度: True

🔄 计算 layer1_linear 层
    输入形状: (4,)
    权重形状: (3, 4)
    偏置形状: (3,)
📦 创建节点 'layer1_linear_output'：
    形状: (3,)
    值: [ 0.08939224  0.09964199 -0.05195787]
    是否需要梯度: True
📦 创建节点 'layer1_activation_output'：
    形状: (3,)
    值: [ 0.08915489  0.09931352 -0.05191116]
    是否需要梯度: True

🔄 计算 layer2_linear 层
    输入形状: (3,)
    权重形状: (2, 3)
    偏置形状: (2,)
📦 创建节点 'layer2_linear_output'：
   

## 💾 内存分析：嵌入式系统约束

### 内存预算分析

在 1KB 的限制下，我们需要仔细计算每一字节：

**参数存储：**
- 权重：W1 (3×4) + W2 (2×3) = 18 个 float32 值 = 72 bytes
- 偏置：b1 (3) + b2 (2) = 5 个 float32 值 = 20 bytes
- **总参数量：92 bytes**

**激活存储（前向传播）：**
- 输入：4 个 float32 = 16 bytes
- 隐层：3 个 float32 = 12 bytes
- 输出：2 个 float32 = 8 bytes
- **总激活量：36 bytes**

**梯度存储（反向传播）：**
- 与参数存储相同：92 bytes

**临时空间：**
- 临时计算：约 50 bytes

**总计：约 270 bytes（远低于 1KB！）**

In [7]:
def analyze_memory_usage():
    print("💾 综合内存分析")
    print("=" * 45)
    all_nodes = [
        input_node, target_node, W1_node, b1_node, z1_node, a1_node, W2_node, b2_node, z2_node, probs_node, loss_node
    ]
    total_value_bytes = 0
    total_grad_bytes = 0
    for node in all_nodes:
        elements = node.value.size
        value_bytes = elements * 4
        grad_bytes = elements * 4 if node.requires_grad else 0
        total_value_bytes += value_bytes
        total_grad_bytes += grad_bytes
    print(f"已用内存: {total_value_bytes + total_grad_bytes} bytes")
    return total_value_bytes + total_grad_bytes

memory_used = analyze_memory_usage()

💾 综合内存分析
已用内存: 296 bytes


## ⬅️ 反向模式自动微分：反向传播

### 理解反向传播

反向传播是魔法发生的地方！从损失值出发（它是一个标量），我们沿着整个网络反向传播梯度，以计算每个参数应该如何更新。

### 反向传播算法

1. **初始化损失梯度**：设置损失梯度为 1.0（∂Loss/∂Loss = 1）
2. **逆拓扑顺序**：从输出到输入依次访问节点
3. **应用链式法则**：每个节点计算其输入的梯度
4. **累积梯度**：合并来自多个路径的贡献

### 为什么这可行

微积分中的链式法则：**∂Loss/∂param = ∂Loss/∂output × ∂output/∂param**

每个反向函数负责实现第二项，而反向传播则自动处理第一项——通过梯度沿图反向传播。

In [8]:
print("⬅️ 反向模式自动微分")
print("=" * 50)
loss_node.gradient = np.array(1.0, dtype=np.float32)
backward_nodes = [loss_node, probs_node, z2_node, a1_node, z1_node]
for node in backward_nodes:
    if node.backward_fn:
        node.backward_fn()

print("🎉 反向传播完成！")
print("所有参数梯度已自动计算完成！")

⬅️ 反向模式自动微分

⬅️ layer2_linear 的反向传播

⬅️ layer1_linear 的反向传播
🎉 反向传播完成！
所有参数梯度已自动计算完成！


## 🎉 练习 2 总结：主要成果

### 我们构建了什么

✅ **完整的计算图实现**
- 基于节点架构的自动微分
- 前向传播及有策略的中间值保存
- 反向传播与自动梯度计算

✅ **现实的唤醒词检测网络**
- 4 个音频特征 → 3 个隐藏单元 → 2 个类别
- 正确的激活函数（tanh、softmax）
- 用于分类的交叉熵损失

✅ **内存高效设计**
- 总内存使用约 270 bytes（远低于 1KB 限制）
- 只保留反向传播所必需的值
- 适合嵌入式系统实现

### 现实世界应用

这个实现展示了背后核心技术：
- **智能音箱**（Alexa、Google Home、Siri）
- **移动语音助手**
- **带语音控制功能的边缘 AI 设备**
- **保护隐私的联邦学习系统**